# Controllable Generative AI for Sequence Design

### Steering a peptide language model toward antimicrobial activity

**ML in PL 2025 — hands-on tutorial (25 min)**

---

Generative models can write plausible peptides all day. That is not the hard part.
The hard part is writing peptides that are *plausible **and** do what you want*.

This tutorial is about that gap, and about the cheapest honest way to close it.

**Roadmap**

| § | Topic | ~min |
|---|-------|------|
| 0 | Setup — install, download checkpoints | 2 |
| 1 | The design problem: why not just enumerate? | 2 |
| 2 | Unconditional generation | 3 |
| 3 | Knob 1 — decoding parameters (temperature, top-k, top-p) | 3 |
| 4 | Knob 2 — a property predictor in the same model | 3 |
| 5 | Knob 3 — best-of-k selection | 4 |
| 6 | Knob 4 — fine-tune the generator on the winners | 5 |
| 7 | Did it actually work? Evaluation and failure modes | 3 |

**The model.** [Hyformer](https://github.com/Pankhil07/AMPFormer) is a *joint*
transformer: one shared backbone with two heads — a language-modelling head that
*generates* sequences and a prediction head that *scores* them. A task token at
position 0 selects which. That single property is what makes the whole tutorial
work: the thing that proposes and the thing that judges are the same network.

**Domain.** Antimicrobial peptides (AMPs) — short peptides that kill bacteria.
They are of interest because antimicrobial resistance is eroding our existing
antibiotics. Our target property is **MIC** (minimum inhibitory concentration)
against *Escherichia coli*, in log₂ µM. **Lower = more potent.**

Both checkpoints are public on HuggingFace and the method is described in
[Izdebski et al., 2025](https://arxiv.org/abs/2504.16559).

---
## 0 — Setup

Runs on a Colab T4. Everything also works on CPU, just slower — reduce
`POOL_SIZE` in §2 if you are on CPU.

In [ ]:
# --- Install -------------------------------------------------------------
import os, sys, subprocess, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Pankhil07/mlinpl-peptide-tutorial.git"   # <-- set to your repo
BRANCH   = "main"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")

def sh(*args):
    print("$", " ".join(args))
    r = subprocess.run(args, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}):\n{r.stderr.strip()}")

if IN_COLAB:
    # The cwd may be a directory that no longer exists (a previous run chdir'd
    # into the clone and it was since deleted), which makes git fail with a
    # confusing exit 128. Step onto solid ground first.
    try:
        os.getcwd()
    except OSError:
        pass
    os.chdir("/content")
    repo = pathlib.Path("/content") / REPO_DIR
    if (repo / ".git").is_dir():
        # Refresh an existing clone. Colab keeps /content across re-runs, so
        # without this you stay pinned to whatever you cloned the first time.
        sh("git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH)
        sh("git", "-C", str(repo), "reset", "--hard", f"origin/{BRANCH}")
        sh("git", "-C", str(repo), "clean", "-fd")
    else:
        shutil.rmtree(repo, ignore_errors=True)
        sh("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo))
    os.chdir(repo)
    print("at commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                       capture_output=True, text=True).stdout.strip())
    # setup.py declares no install_requires; deps are listed explicitly here.
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".")
    sh(sys.executable, "-m", "pip", "install", "-q", "einops", "huggingface_hub")

print("cwd:", os.getcwd())

In [ ]:
# --- Download checkpoints from HuggingFace -------------------------------
# Two public checkpoints (BSD-3-Clause), ~610 MB total:
#   GEN_CKPT - Hyformer pretrained on 3.5M general + antimicrobial peptides
#   MIC_CKPT - the same backbone jointly fine-tuned to predict log2 MIC
from huggingface_hub import hf_hub_download

GEN_REPO = "SzczurekLab/hyformer_peptides_34M"
MIC_REPO = "SzczurekLab/hyformer_peptides_34M_MIC"

GEN_CKPT = hf_hub_download(GEN_REPO, "ckpt.pt")
MIC_CKPT = hf_hub_download(MIC_REPO, "ckpt.pt")

for name, p in [("generator", GEN_CKPT), ("MIC predictor", MIC_CKPT)]:
    print(f"{name:<14} {os.path.getsize(p) / 1e6:7.1f} MB")

In [ ]:
# --- Imports and global config -------------------------------------------
import os
import pathlib

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import Counter

from hyformer.configs.model import ModelConfig
from hyformer.configs.tokenizer import TokenizerConfig
from hyformer.configs.dataset import DatasetConfig
from hyformer.models.auto import AutoModel
from hyformer.utils.tokenizers.auto import AutoTokenizer

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


# --- Locate the repo root --------------------------------------------------
# The bundled configs store paths relative to the repo root (e.g. the tokenizer
# config points at "data/vocabulary/aa.txt"), so they only resolve if the cwd
# is right. Find the root explicitly instead of assuming it.
def _find_repo_root():
    marker = pathlib.Path("data") / "vocabulary" / "aa.txt"
    here = pathlib.Path.cwd()

    candidates = []
    # Most reliable anchor: the installed hyformer package lives in the repo.
    try:
        import hyformer
        candidates.append(pathlib.Path(hyformer.__file__).resolve().parent.parent)
    except Exception:
        pass
    candidates += [here, *here.parents, *sorted(pathlib.Path("/content").glob("*"))]

    for c in candidates:
        if (c / marker).is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the tutorial repo root (no data/vocabulary/aa.txt near "
        f"{here}). Run the setup cell at the top of this notebook first."
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
print("repo root:", REPO_ROOT)

MODEL_CFG    = str(REPO_ROOT / "configs/models/hyformer/config.json")
TOK_CFG      = str(REPO_ROOT / "configs/tokenizers/amino_acid/config.json")
MIC_DSET_CFG = str(REPO_ROOT / "configs/datasets/ab_mic/config.json")


def load_tokenizer():
    """Build the AA tokenizer, resolving its vocabulary path against the root."""
    cfg = TokenizerConfig.from_config_filepath(TOK_CFG)
    cfg.vocabulary_path = str(REPO_ROOT / cfg.vocabulary_path)
    return AutoTokenizer.from_config(cfg)

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

# --- Plot style ----------------------------------------------------------
# Palette roles: categorical slots 1-2 for "all vs selected"; a single-hue
# ordinal blue ramp for temperature (an ordered quantity, not an identity).
C_ALL, C_SEL, C_THIRD = "#2a78d6", "#eb6834", "#1baf7a"
RAMP_BLUE = ["#86b6ef", "#2a78d6", "#104281"]
INK, INK_MUTED = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.dpi": 120, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#d8d7d2", "axes.linewidth": 0.8, "axes.labelcolor": INK_MUTED,
    "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#eceae5", "grid.linewidth": 0.8, "axes.grid": True, "axes.axisbelow": True,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED, "font.size": 9,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 8,
})
print("ready")

---
## 1 — The design problem

A peptide of length $L$ over the 20 natural amino acids has $20^L$ possible
sequences. Let's put a number on that.

In [ ]:
lengths = np.arange(5, 51)
space = 20.0 ** lengths

for L in (10, 20, 30, 50):
    print(f"L = {L:2d}  ->  20^{L} = {20.0**L:.2e} sequences")

print(f"\nAtoms in the observable universe: ~1e80")
print(f"Peptides of length 50:            ~{20.0**50:.0e}")

fig, ax = plt.subplots(figsize=(6.0, 3.0))
ax.plot(lengths, space, color=C_ALL)
ax.axhline(1e80, color=INK_MUTED, linestyle="--", linewidth=1)
ax.annotate("atoms in the observable universe", xy=(7, 2e76),
            color=INK_MUTED, fontsize=8, va="top")
ax.set_yscale("log")
ax.set_ylim(1e5, 1e86)
ax.set_xlabel("peptide length (residues)")
ax.set_ylabel("possible sequences")
ax.set_title("Sequence space defeats enumeration", loc="left")
plt.tight_layout(); plt.show()

Enumeration is off the table, and random sampling is worse than useless —
the overwhelming majority of those strings are not even peptide-like, let
alone active.

So the design problem splits in two:

1. **Propose** — a generative model that puts its probability mass on
   *plausible* peptides, shrinking $20^L$ to something navigable.
2. **Score** — a predictor that estimates the property we care about.

**Control** is how we couple them. The rest of this tutorial is three
increasingly targeted ways to do that coupling.

---
## 2 — Unconditional generation

Load the pretrained Hyformer and sample from it with no steering at all.

Generation is prefixed by two tokens: `[<lm>, <s>]` — the task token that puts
the shared backbone in language-modelling mode, then beginning-of-sequence.

In [ ]:
tokenizer = load_tokenizer()
gen_model = AutoModel.from_config(ModelConfig.from_config_filepath(MODEL_CFG))

ckpt = torch.load(GEN_CKPT, map_location=DEVICE)
gen_model.load_pretrained(state_dict=ckpt.get("model", ckpt), discard_prediction_head=True)
gen_model.to(DEVICE).eval()

LM_ID  = tokenizer.task_token_id("lm")
BOS_ID = tokenizer.bos_token_id
EOS_ID = tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id
print(f"task token <lm>={LM_ID}  bos={BOS_ID}  eos={EOS_ID}  pad={PAD_ID}")

In [ ]:
def is_valid(seq, min_len=5, max_len=50):
    return bool(seq) and min_len <= len(seq) <= max_len and all(c in VALID_AA for c in seq)


@torch.inference_mode()
def sample_peptides(n, temperature=1.0, top_k=None, top_p=None,
                    max_len=50, batch_size=256, dedup=True, max_rounds=200):
    """Sample `n` valid peptides from the LM head. Returns a list of strings."""
    prefix = torch.tensor([[LM_ID, BOS_ID]], dtype=torch.long, device=DEVICE)
    out, seen = [], set()
    for _ in range(max_rounds):
        if len(out) >= n:
            break
        bs = min(batch_size, max(32, (n - len(out)) * 2))
        ids = gen_model.generate(
            prefix_input_ids=prefix.expand(bs, -1),
            num_tokens_to_generate=max_len,
            eos_token_id=EOS_ID, pad_token_id=PAD_ID,
            temperature=temperature, top_k=top_k, top_p=top_p,
            use_cache=False,
        )
        # NB: tokenizer.decode() on a 2-D tensor silently decodes row 0 only,
        # so decode row by row. (Same reason we don't use model.to_generator().)
        for row in ids:
            s = tokenizer.decode(row.tolist(), skip_special_tokens=True).strip().upper()
            if not is_valid(s, max_len=max_len):
                continue
            if dedup and s in seen:
                continue
            seen.add(s); out.append(s)
            if len(out) >= n:
                break
    if len(out) < n:
        print(f"[!] only reached {len(out)}/{n} after {max_rounds} rounds")
    return out[:n]

In [ ]:
POOL_SIZE = 2000          # lower to ~400 if you are on CPU
%time pool = sample_peptides(POOL_SIZE, temperature=1.0)

print(f"\n{len(pool)} unique valid peptides\n")
for s in pool[:8]:
    print(f"  {s}   (len {len(s)})")

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 2.8))
ax.hist([len(s) for s in pool], bins=range(5, 52, 2), color=C_ALL)
ax.set_xlabel("length (residues)"); ax.set_ylabel("count")
ax.set_title(f"Length distribution of {len(pool)} unconditional samples")
plt.tight_layout(); plt.show()

These read like peptides. Amino-acid usage is non-uniform, lengths are
sensible, nothing is obviously broken.

**And we have no idea whether a single one of them is active.** The model was
trained to reproduce a distribution of peptides, not to maximise potency.
Plausible ≠ useful. That is the whole problem.

---
## 3 — Knob 1: decoding parameters

The first knobs anyone reaches for are the sampling parameters:

| knob | what it does |
|------|--------------|
| **temperature** | rescales logits. <1 sharpens toward the mode, >1 flattens |
| **top-k** | sample only from the k highest-probability tokens |
| **top-p** (nucleus) | sample from the smallest set whose mass exceeds p |

Watch what they control — and what they don't.

In [ ]:
def kmer_set(s, k=3):
    return {s[i:i + k] for i in range(len(s) - k + 1)}


def internal_diversity(seqs, n_pairs=4000, k=3, rng=None):
    """Mean pairwise 3-mer Jaccard distance over random pairs. 1.0 = all different."""
    rng = rng or np.random.default_rng(0)
    if len(seqs) < 2:
        return float("nan")
    sets = [kmer_set(s, k) for s in seqs]
    i = rng.integers(0, len(seqs), n_pairs)
    j = rng.integers(0, len(seqs), n_pairs)
    d = []
    for a, b in zip(i, j):
        if a == b:
            continue
        inter = len(sets[a] & sets[b]); union = len(sets[a] | sets[b])
        if union:
            d.append(1.0 - inter / union)
    return float(np.mean(d))


TEMPS = [0.7, 1.0, 1.3]
N_SWEEP = 400
sweep = {}
for T in TEMPS:
    s = sample_peptides(N_SWEEP, temperature=T, dedup=False)
    sweep[T] = s
    print(f"T={T}:  unique={len(set(s)) / len(s):.3f}   "
          f"mean_len={np.mean([len(x) for x in s]):5.1f}   "
          f"diversity={internal_diversity(s):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.0))
for T, c in zip(TEMPS, RAMP_BLUE):
    ax.hist([len(s) for s in sweep[T]], bins=range(5, 52, 3),
            histtype="step", linewidth=2, color=c, label=f"T = {T}")
ax.set_xlabel("length (residues)"); ax.set_ylabel("count")
ax.set_title("Temperature reshapes the distribution")
ax.legend(title="temperature", title_fontsize=8)
plt.tight_layout(); plt.show()

Temperature buys us a **diversity ↔ conservatism** trade: low T concentrates on
the model's favourite patterns, high T spreads out and starts producing junk.

> **The key limitation.** Not one of these knobs has any idea what MIC is.
> They reshape the model's *own* distribution. They cannot move it toward a
> property the model was never told about. For that we need a second signal.

---
## 4 — Knob 2: a property predictor in the same model

Hyformer's prediction head shares the backbone with the LM head. Swap the task
token at position 0 from `<lm>` to `<prediction>` and the same weights that
wrote the sequence now score it.

The head here was fine-tuned to regress **log₂(MIC / µM)** against
*E. coli*. Lower is more potent.

> **Keep this number in view for the rest of the notebook.** The checkpoint's
> best validation MSE is **0.78**, i.e. a typical error of about **±0.9 log₂
> units** — nearly a two-fold error in MIC. Every ranking below is built on a
> signal with that much noise in it.

In [ ]:
dset_cfg = DatasetConfig.from_config_filepath(MIC_DSET_CFG)
mic_model = AutoModel.from_config(
    ModelConfig.from_config_filepath(MODEL_CFG),
    prediction_task_type=dset_cfg.prediction_task_type,
    num_prediction_tasks=dset_cfg.num_prediction_tasks,
)
mic_ckpt = torch.load(MIC_CKPT, map_location=DEVICE)
mic_model.load_pretrained(state_dict=mic_ckpt.get("model", mic_ckpt),
                          discard_prediction_head=False)
mic_model.to(DEVICE).eval()

PRED_ID = tokenizer.task_token_id("prediction")
print(f"task token <prediction>={PRED_ID}  "
      f"(task={dset_cfg.prediction_task_type}, n_tasks={dset_cfg.num_prediction_tasks})")

In [ ]:
@torch.inference_mode()
def predict_log2_mic(seqs, batch_size=128):
    """Predicted log2(MIC / uM) for each sequence. Lower = more potent."""
    preds = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i + batch_size]
        ids_list = tokenizer(batch, task="prediction")["input_ids"]
        m = max(len(x) for x in ids_list)
        padded = [x + [PAD_ID] * (m - len(x)) for x in ids_list]
        input_ids = torch.tensor(padded, dtype=torch.long, device=DEVICE)
        # Layout is [<task>, <s>, ...residues..., </s>] and the prediction head
        # reads position 0, so position 0 must hold <cls>. Belt-and-braces: the
        # tokenizer already put it there, but generated ids carry <lm> instead.
        input_ids[:, 0] = PRED_ID
        attn = (input_ids != PAD_ID).long()
        out = mic_model(input_ids=input_ids, attention_mask=attn,
                        task="prediction", return_loss=False)
        preds.append(out["logits"].float().cpu().numpy()[:, 0])
    return np.concatenate(preds)


# Sanity check on well-characterised natural AMPs before we trust it on samples.
KNOWN_AMPS = {
    "Magainin-2":  "GIGKFLHSAKKFGKAFVGEIMNS",
    "Melittin":    "GIGAVLKVLTTGLPALISWIKRKRQQ",
    "LL-37 (frag)":"KRIVQRIKDFLRNLVPRTES",
    "Buforin-II":  "TRSSRAGLQFPVGRVHRLLRK",
}
for name, seq in KNOWN_AMPS.items():
    v = predict_log2_mic([seq])[0]
    print(f"  {name:<14} log2MIC = {v:6.2f}   ->  {2.0 ** v:8.2f} uM")

In [ ]:
%time mic_pool = predict_log2_mic(pool)

print(f"predicted log2 MIC over the pool:")
print(f"  mean   {mic_pool.mean():.2f}")
print(f"  median {np.median(mic_pool):.2f}")
print(f"  min    {mic_pool.min():.2f}  (most potent)")
print(f"  max    {mic_pool.max():.2f}")

fig, ax = plt.subplots(figsize=(5.2, 2.8))
ax.hist(mic_pool, bins=40, color=C_ALL)
ax.set_xlabel("predicted log$_2$(MIC / µM)   ←  more potent")
ax.set_ylabel("count")
ax.set_title("The unsteered pool contains a potent tail — but mostly mediocrity")
plt.tight_layout(); plt.show()

---
## 5 — Knob 3: best-of-k selection

Now the simplest method that actually targets the property:

> **Generate $k$, keep the best.**

This is rejection sampling, and it is the baseline every fancier method has to
beat. It needs no retraining, no gradients, no architecture changes. It only
needs a generator and a scorer — which we have.

Formally we are reweighting the generator's distribution by the predictor,
approximating $p(x \mid \text{potent}) \propto p_\text{LM}(x)\,\cdot\,
\mathbb{1}[\hat{y}(x) < \tau]$ by brute force.

In [ ]:
KEEP_FRAC = 0.05                      # keep the top 5% most potent

order = np.argsort(mic_pool)          # ascending: most potent first
n_keep = max(1, int(len(pool) * KEEP_FRAC))
sel_idx = order[:n_keep]

selected = [pool[i] for i in sel_idx]
mic_sel = mic_pool[sel_idx]

print(f"kept {n_keep} / {len(pool)} sequences (top {KEEP_FRAC:.0%})\n")
print(f"  mean log2MIC  pool     {mic_pool.mean():6.2f}")
print(f"  mean log2MIC  selected {mic_sel.mean():6.2f}"
      f"   (shift {mic_sel.mean() - mic_pool.mean():+.2f})")
print(f"\n  in MIC terms: {2**mic_pool.mean():.1f} uM  ->  {2**mic_sel.mean():.1f} uM")

print("\nTop 10 by predicted potency:")
for i in order[:10]:
    print(f"  {pool[i]:<36} log2MIC={mic_pool[i]:6.2f}  ({2**mic_pool[i]:7.2f} uM)")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.0))
bins = np.linspace(mic_pool.min(), mic_pool.max(), 45)
ax.hist(mic_pool, bins=bins, color=C_ALL, label=f"all ({len(pool)})")
ax.hist(mic_sel, bins=bins, color=C_SEL, label=f"selected ({n_keep})")
ax.axvline(mic_pool.mean(), color=C_ALL, linestyle="--", linewidth=1.5)
ax.axvline(mic_sel.mean(), color=C_SEL, linestyle="--", linewidth=1.5)
ax.set_xlabel("predicted log$_2$(MIC / µM)   ←  more potent")
ax.set_ylabel("count")
ax.set_title(f"Best-of-k moves the distribution (dashed = means)")
ax.legend()
plt.tight_layout(); plt.show()

### Exercise

Change `KEEP_FRAC` below and rerun. Try `0.5`, `0.1`, `0.01`, `0.002`.

Two questions to hold in mind:
- How much does the mean predicted potency improve?
- How many *distinct* sequences are you left with to hand to a chemist?

In [ ]:
fracs = np.array([0.5, 0.25, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002])
mean_mic, divers, n_left = [], [], []
for f in fracs:
    k = max(2, int(len(pool) * f))
    idx = order[:k]
    mean_mic.append(mic_pool[idx].mean())
    divers.append(internal_diversity([pool[i] for i in idx]))
    n_left.append(k)

# Two measures on different scales -> two panels sharing an x-axis, never two y-axes.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.2, 4.6), sharex=True)
fig.suptitle("Selection pressure: what you gain, and what you pay",
             x=0.02, ha="left", fontsize=11, fontweight="semibold", color=INK)

ax1.plot(fracs, mean_mic, color=C_SEL, marker="o")
ax1.set_ylabel("mean predicted log$_2$MIC")
ax1.invert_yaxis()                      # up = more potent
ax1.margins(y=0.22)
ax1.annotate("↑ more potent", xy=(0.70, 0.88), xycoords="axes fraction",
             color=INK_MUTED, fontsize=8)

ax2.plot(fracs, divers, color=C_THIRD, marker="o")
ax2.set_ylabel("internal diversity")
ax2.set_xlabel("keep fraction (log scale)")
ax2.set_xscale("log")
ax2.set_ylim(0, 1.05)
for f, d, n in zip(fracs, divers, n_left):
    if f in (fracs[0], fracs[-1]):
        ax2.annotate(f"n={n}", xy=(f, d), xytext=(0, 10),
                     textcoords="offset points", ha="center",
                     color=INK_MUTED, fontsize=8)
plt.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

There is the trade, and it is the central fact about best-of-k: **potency is
bought with diversity and with compute.** Push the keep-fraction down and the
survivors get better-scoring and more alike, until you are holding a handful of
near-duplicates. To keep both you must generate a bigger pool — cost grows
roughly as $1/\text{keep fraction}$.

---
## 6 — Knob 4: fine-tune the generator on the winners

Best-of-k has a hard ceiling: **you only ever get the $k$ you sampled.** Want
another hundred good peptides? Pay the whole generate-and-score bill again. The
model never learned anything.

So move the knowledge *into the generator*: take the winners and fine-tune on
them. The model's distribution shifts toward the region the predictor likes, and
then you can sample from it as much as you want.

This is **rejection-sampling fine-tuning** — also called RAFT, reward-weighted
regression, or hill-climbing. One round below; real pipelines loop
generate → score → select → fine-tune for many iterations.

```
  sample  ──▶  score  ──▶  keep top k%  ──▶  fine-tune LM  ──┐
     ▲                                                        │
     └────────────────────────────────────────────────────────┘
```

In [ ]:
import copy

FT_FRAC   = 0.10      # train on the top 10% (more data than we selected in §5)
FT_EPOCHS = 6
FT_LR     = 3e-5
FT_BATCH  = 32

ft_idx  = order[:max(8, int(len(pool) * FT_FRAC))]
ft_seqs = [pool[i] for i in ft_idx]
print(f"fine-tuning on {len(ft_seqs)} sequences  "
      f"(mean log2MIC {mic_pool[ft_idx].mean():.2f} vs pool {mic_pool.mean():.2f})")

# Work on a COPY so the original generator stays available for comparison.
ft_model = copy.deepcopy(gen_model).to(DEVICE)
ft_model.train()
opt = torch.optim.AdamW(ft_model.parameters(), lr=FT_LR)


def lm_batch(seqs):
    """Tokenize with the <lm> task token and build shifted-LM labels."""
    enc = tokenizer(seqs, task="lm")["input_ids"]
    L = max(len(x) for x in enc)
    ids = torch.tensor([x + [PAD_ID] * (L - len(x)) for x in enc],
                       dtype=torch.long, device=DEVICE)
    labels = ids.clone()
    labels[labels == PAD_ID] = -100        # don't train on padding
    return ids, labels


# Note: for task='lm' the model forces causal attention and ignores
# attention_mask, so trailing pads can't leak into real positions. Masking the
# pad *labels* is what keeps them out of the loss.
rng_ft = np.random.default_rng(SEED)
for epoch in range(FT_EPOCHS):
    perm, ep_loss, nb = rng_ft.permutation(len(ft_seqs)), 0.0, 0
    for i in range(0, len(perm), FT_BATCH):
        batch = [ft_seqs[j] for j in perm[i:i + FT_BATCH]]
        if len(batch) < 2:
            continue
        ids, labels = lm_batch(batch)
        loss = ft_model(input_ids=ids, task="lm",
                        input_labels=labels, return_loss=True)["loss"]
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)   # grads are large
        opt.step()
        ep_loss += loss.item(); nb += 1
    print(f"  epoch {epoch + 1}/{FT_EPOCHS}   loss {ep_loss / max(nb, 1):.4f}")

ft_model.eval()
print("done")

In [ ]:
# Sample FRESH sequences from the fine-tuned model and score them with the
# SAME (frozen) predictor, so the comparison is apples to apples.
_orig = gen_model
try:
    gen_model = ft_model                     # sample_peptides() closes over gen_model
    ft_pool = sample_peptides(600, temperature=1.0)
finally:
    gen_model = _orig

mic_ft = predict_log2_mic(ft_pool)

print(f"{'set':<28}{'n':>6}{'mean log2MIC':>15}{'mean MIC (uM)':>16}")
print("-" * 65)
for name, seqs, vals in [("original pool", pool, mic_pool),
                         ("best-of-k selected", selected, mic_sel),
                         ("after fine-tuning", ft_pool, mic_ft)]:
    print(f"{name:<28}{len(seqs):>6}{vals.mean():>15.2f}{2.0 ** vals.mean():>16.2f}")

print(f"\nshift from fine-tuning: {mic_ft.mean() - mic_pool.mean():+.2f} log2 units")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.1))
bins = np.linspace(min(mic_pool.min(), mic_ft.min()),
                   max(mic_pool.max(), mic_ft.max()), 50)
ax.hist(mic_pool, bins=bins, color=C_ALL, alpha=0.95, label="original pool")
ax.hist(mic_ft, bins=bins, color=C_THIRD, alpha=0.80, label="after fine-tuning")
for v, c in [(mic_pool.mean(), C_ALL), (mic_ft.mean(), C_THIRD)]:
    ax.axvline(v, color=c, linestyle="--", linewidth=1.5)
ax.set_xlabel("predicted log$_2$(MIC / µM)   ←  more potent")
ax.set_ylabel("count")
ax.set_title("Fine-tuning moves the whole distribution, not just a slice", loc="left")
ax.legend()
plt.tight_layout(); plt.show()

That is the difference worth internalising. Best-of-k moved *a slice* — it threw
95% of the pool away. Fine-tuning moved **the distribution itself**, so every
future sample is drawn from the better region, at no extra selection cost.

Now the bill.

In [ ]:
# How much of this is genuine generalisation, and how much is memorisation?
ft_train_set = set(ft_seqs)
copied = sum(s in ft_train_set for s in ft_pool)

print(f"{'metric':<34}{'pool':>10}{'fine-tuned':>13}")
print("-" * 57)
print(f"{'unique fraction':<34}{len(set(pool)) / len(pool):>10.3f}"
      f"{len(set(ft_pool)) / len(ft_pool):>13.3f}")
print(f"{'internal diversity':<34}{internal_diversity(pool):>10.3f}"
      f"{internal_diversity(ft_pool):>13.3f}")
print(f"{'mean length':<34}{np.mean([len(s) for s in pool]):>10.1f}"
      f"{np.mean([len(s) for s in ft_pool]):>13.1f}")
print(f"\nverbatim copies of the fine-tuning set: {copied} / {len(ft_pool)}"
      f"  ({copied / len(ft_pool):.1%})")
print(f"(high % = the model memorised its {len(ft_seqs)} examples "
      f"instead of learning the region around them)")

**Read those two numbers together.** A potency gain bought entirely by
collapsing onto 200 memorised training sequences is not a design method — it is
an expensive lookup table. What you want is the distribution shifted *with*
diversity largely intact, and verbatim copies near zero.

Levers when it collapses: fewer epochs, lower learning rate, a larger / less
aggressively filtered fine-tuning set, or an explicit KL penalty against the
original model (the standard fix, and what RLHF pipelines use).

> **And the deeper problem:** the predictor was frozen throughout. We just
> trained the generator to satisfy it. Wherever that predictor is wrong, the
> error is now baked into the generator — and the next round of selection will
> amplify it. Iterating this loop without new ground-truth measurements is a
> machine for converging on your proxy's blind spots.

---
## 7 — Did it actually work?

A distribution shift in a predicted score is *not* evidence of success. Four
standard checks, then the failure modes that matter more than any of them.

In [ ]:
def report(name, seqs):
    return {
        "set": name,
        "n": len(seqs),
        "validity": np.mean([is_valid(s) for s in seqs]),
        "uniqueness": len(set(seqs)) / len(seqs),
        "diversity": internal_diversity(seqs),
        "mean_len": float(np.mean([len(s) for s in seqs])),
    }

rows = [report("pool", pool), report("selected", selected), report("fine-tuned", ft_pool)]
hdr = f"{'set':<12}{'n':>6}{'validity':>10}{'uniqueness':>12}{'diversity':>11}{'mean_len':>10}"
print(hdr); print("-" * len(hdr))
for r in rows:
    print(f"{r['set']:<12}{r['n']:>6}{r['validity']:>10.3f}"
          f"{r['uniqueness']:>12.3f}{r['diversity']:>11.3f}{r['mean_len']:>10.1f}")

In [ ]:
# Composition shift: what is selection actually latching onto?
def aa_freq(seqs):
    c = Counter("".join(seqs))
    tot = sum(c.values())
    return np.array([c.get(a, 0) / tot for a in "ACDEFGHIKLMNPQRSTVWY"])

f_pool, f_sel = aa_freq(pool), aa_freq(selected)
AAS = list("ACDEFGHIKLMNPQRSTVWY")
delta = f_sel - f_pool                     # the quantity the story is about
x = np.arange(20)

# Polarity (enriched vs depleted) -> a diverging encoding around zero,
# not two bars the eye has to subtract by itself.
C_POS, C_NEG = "#2a78d6", "#e34948"
fig, ax = plt.subplots(figsize=(6.6, 3.0))
ax.bar(x, delta, width=0.62, color=[C_POS if d >= 0 else C_NEG for d in delta])
ax.axhline(0, color=INK_MUTED, linewidth=0.9)
ax.set_xticks(x); ax.set_xticklabels(AAS)
ax.set_ylabel("Δ frequency\n(selected − pool)"); ax.set_xlabel("amino acid")
ax.set_title("Which residues selection enriches", loc="left")
for i, d in enumerate(delta):             # label only the extremes
    if abs(d) >= np.sort(np.abs(delta))[-4]:
        ax.annotate(AAS[i], xy=(i, d), xytext=(0, 4 if d >= 0 else -11),
                    textcoords="offset points", ha="center",
                    color=INK, fontsize=8, fontweight="semibold")
plt.tight_layout(); plt.show()

charge = lambda s: sum(s.count(a) for a in "KR") - sum(s.count(a) for a in "DE")
print(f"mean net charge   pool {np.mean([charge(s) for s in pool]):+.2f}"
      f"   selected {np.mean([charge(s) for s in selected]):+.2f}")

Expect the selected set to be enriched in **lysine (K)** and in hydrophobic
residues such as **L** and **F**, with mean net charge rising sharply. That is
the textbook AMP motif — cationic and amphipathic, so the peptide binds anionic
bacterial membranes — and recovering it is a genuinely encouraging sign that the
predictor learned real biochemistry rather than a surface artefact.

Note that **arginine (R) usually does *not* come along for the ride**, even
though it carries the same +1 charge. The model has learned something more
specific than "positive is good". Whether that reflects real biology or a bias
in the training set is exactly the kind of question this method cannot answer
for you.

It is also exactly what a predictor would learn if it had memorised *"charged =
active"* and nothing else. Which brings us to the part that matters.

In [ ]:
# A deliberately adversarial probe: does the predictor know what it doesn't know?
PROBES = {
    "poly-K (nonsense)":      "KKKKKKKKKKKKKKKKKKKK",
    "poly-A (inert)":         "AAAAAAAAAAAAAAAAAAAA",
    "poly-E (anionic)":       "EEEEEEEEEEEEEEEEEEEE",
    "Magainin-2 (real AMP)":  KNOWN_AMPS["Magainin-2"],
    "Magainin-2 REVERSED":    KNOWN_AMPS["Magainin-2"][::-1],
    "Magainin-2 SHUFFLED":    "".join(np.random.default_rng(0).permutation(
                                  list(KNOWN_AMPS["Magainin-2"]))),
}
vals = predict_log2_mic(list(PROBES.values()))
print(f"{'probe':<26}{'log2MIC':>9}{'MIC (uM)':>12}")
print("-" * 47)
for (name, _), v in zip(PROBES.items(), vals):
    print(f"{name:<26}{v:>9.2f}{2.0**v:>12.2f}")

### Failure modes — the part to actually remember

1. **The predictor is a proxy, not the truth.** Every number in this notebook is
   a model output. Nothing here has touched a bacterium.

2. **Goodhart's law.** The moment you *select on* the predictor, it stops being
   an unbiased measure. Best-of-k is an adversarial search for the predictor's
   blind spots — the top-scoring sequences are disproportionately likely to be
   the ones where it is wrong.

3. **Distribution shift / OOD.** The MIC head was fine-tuned on measured
   peptides. We are now scoring *generated* ones. The further selection pushes,
   the further out of distribution the survivors are — precisely where the
   predictor is least reliable, and it reports no uncertainty while doing it.
   Look at the poly-K probe above: a confident number for a string no one would
   call a drug candidate.

4. **No uncertainty estimate.** A single scalar with no error bar invites
   over-trust. Ensembles or MC-dropout would at least flag the shaky cases.

5. **Potency is one axis of many.** A real candidate also needs low
   haemolysis and cytotoxicity, protease stability, and manufacturability. A
   peptide optimised for MIC alone will cheerfully be toxic to human cells —
   cationic amphipathic peptides lyse membranes, and human membranes are
   membranes too. Single-objective optimisation is a trap.

**The honest summary:** this notebook produces *hypotheses, ranked*. That is
genuinely useful — it is a vastly better use of a wet-lab budget than random
screening. It is not a discovery. Ground truth is an assay.

---
## 8 — Where to go from here

We covered the first two rows. In rough order of effort:

| method | idea | cost |
|--------|------|------|
| **Best-of-k** *(§5)* | sample many, rank, keep the best | generation only |
| **Rejection-sampling fine-tuning** *(§6)* | fine-tune the generator on the winners, then iterate | generation + a short training run |
| **Guided decoding** (FUDGE) | predictor reweights next-token logits at *every* step, so control is applied during generation rather than after | a predictor over prefixes |
| **Conditional training** | train with the property as a control token; ask for a value at sampling time | retraining + labels |
| **Multi-objective** | optimise potency *and* low haemolysis *and* stability, jointly | several predictors |

**Want to try guided decoding yourself?** The hook is
`Hyformer._generate_single_token_logits(prefix_input_ids, temperature, top_k,
top_p, use_cache)`, which returns `(batch, vocab)` logits for the next token.
Add your own guidance term to those logits and sample — that is FUDGE in about
twenty lines.

In this repo:
- `experiments/notebooks/fudge.ipynb` — guided-decoding experiments
- `experiments/notebooks/99-best-of-k-generation.ipynb` — the method above, at scale
- `finetune.py` — fine-tune the prediction head on *your* sequence-property data

*(Both notebooks predate the current API — `task='generation'` is now `'lm'`,
and the several `logits_*` output keys are now a single `logits`. Treat them as
reference, not as runnable code.)*

### Three things worth taking away

1. A generative model gives you **plausibility**; control requires a **second
   signal** that encodes what you actually want.
2. **Best-of-k is a strong, nearly free baseline** — always run it first. When
   you need more than $k$ sequences, **fine-tuning on the winners moves the
   distribution** instead of just filtering it.
3. Every method here trades **potency against diversity**. Measure both, every
   time. A score that improves while diversity collapses has told you nothing.
4. The bottleneck is almost never the generator. It is **the quality and
   calibration of the predictor**, and your willingness to distrust it.

---

*Tutorial materials: [github.com/Pankhil07/AMPFormer](https://github.com/Pankhil07/AMPFormer) · ML in PL 2025*